# 🤖 Desafio 2 — Regressão, Classificação e Machine Learning

## Cenário

> ⏱️ Trabalho em equipe. Cada parte tem **código base** e **células `# TODO`** para completar.

### Roteiro
1. **Classificação** — prever se um pedido vai atrasar (`Late_delivery_risk`).
2. **Regressão** — prever a receita do pedido (`Sales`).
3. **Feature Importance** — Random Forest / Gradient Boosting: o que mais pesa no atraso?
4. **Clustering** — K-Means em `Latitude`/`Longitude` para mapear regiões de risco.

---


In [ ]:
import piplite; await piplite.install("scikit-learn")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             r2_score, mean_absolute_error, mean_squared_error)

pd.set_option('display.max_columns', None)
plt.rcParams['axes.grid'] = True

df = pd.read_csv('data/supplychain.csv')
print('Dimensões:', df.shape)
df.head()

---
## 🎯 Parte 1 — Classificação: vai atrasar?

A coluna alvo é **`Late_delivery_risk`** (0 = no prazo, 1 = atrasado).
Vamos prever o atraso usando apenas **`Shipping Mode`**, **`Customer Segment`** e
**`Category Name`** (variáveis conhecidas **antes** da entrega — evitando *data leakage*).

> ⚠️ **Não** usem `Days for shipping (real)` nem `Delivery Status`: elas só existem
> *depois* da entrega e dariam um modelo "trapaceiro".

💡 **Código base:** já preparamos X, y e a divisão treino/teste com codificação one-hot das categóricas.

In [ ]:
features_cat = ['Shipping Mode', 'Customer Segment', 'Category Name']
alvo = 'Late_delivery_risk'

X = df[features_cat].copy()
y = df[alvo].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Pré-processador: one-hot nas categóricas
preproc = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore'), features_cat)])

print('Treino:', X_train.shape, '| Teste:', X_test.shape)
print('Proporção de atrasos (y):', round(y.mean(), 3))

X_train_encoded = preproc.fit_transform(X_train)
X_test_encoded = preproc.transform(X_test)

### 🎯 O Desafio
1. Treinem um classificador (sugestão: `LogisticRegression` ou `DecisionTreeClassifier`)
2. Avaliem com **acurácia** e **matriz de confusão**.
3. **Discussão:** o que é pior para a empresa — um **Falso Positivo** (prever atraso que
   não aconteceu) ou um **Falso Negativo** (não prever um atraso que aconteceu)?

In [ ]:
# TODO — montar o Pipeline (preproc + modelo) e treinar
# modelo = Pipeline([('prep', preproc), ('clf', LogisticRegression(max_iter=1000))])
# modelo.fit(X_train, y_train)

# TODO — prever no teste, calcular acurácia

# TODO — plotar a matriz de confusão (ConfusionMatrixDisplay)


---
## 📈 Parte 2 — Regressão: prevendo a receita

Vamos prever **`Sales`** (valor de venda do pedido) — útil para previsão de demanda/receita.
Usaremos preditores plausíveis: `Order Item Quantity`, `Order Item Product Price`,
`Order Item Discount`, além de `Category Name` e `Customer Segment`.

> ⚠️ Evitem usar `Order Item Total`, `Sales per customer` e `Order Profit Per Order`:
> são quase a mesma coisa que o alvo (*leakage*).

In [ ]:
num_feats = ['Order Item Quantity', 'Order Item Product Price', 'Order Item Discount']
cat_feats = ['Category Name', 'Customer Segment']
alvo_reg = 'Sales'

Xr = df[num_feats + cat_feats].copy()
yr = df[alvo_reg].copy()

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.25, random_state=42)

preproc_reg = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats)])

print('Treino:', Xr_train.shape, '| Teste:', Xr_test.shape)
Xr_train_prep = preproc_reg.fit_transform(Xr_train)
Xr_test_prep  = preproc_reg.transform(Xr_test)

### 🎯 O Desafio
1. Treinem uma `LinearRegression` num `Pipeline` com `preproc_reg`.
2. Avaliem com **R²**, **MAE** e **RMSE**.
3. Plotem **valor real × valor previsto** e comentem onde o modelo erra mais.

In [ ]:
# TODO — Pipeline de regressão (preproc_reg + LinearRegression) e treino

# TODO — prever e calcular R², MAE, RMSE

# TODO — scatter: y_real (x) vs y_previsto (y) + linha de referência (opcional)


---
## 🌲 Parte 3 — Feature Importance (Random Forest / Gradient Boosting)

Modelos de árvore conseguem dizer **quais variáveis mais pesaram** na previsão.
Vamos voltar ao alvo `Late_delivery_risk`, agora com **mais variáveis** e um modelo mais forte.

In [ ]:
feats_imp_cat = ['Shipping Mode', 'Customer Segment', 'Category Name',
                 'Order Region', 'Department Name', 'Market']
feats_imp_num = ['Days for shipment (scheduled)', 'Order Item Quantity',
                 'Order Item Discount']

Xi = df[feats_imp_cat + feats_imp_num].copy()
yi = df['Late_delivery_risk'].copy()

Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    Xi, yi, test_size=0.25, random_state=42, stratify=yi)

preproc_imp = ColumnTransformer([
    ('num', 'passthrough', feats_imp_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feats_imp_cat)])

print('Pronto. Variáveis:', Xi.shape[1], 'colunas brutas')

Xi_train_prep = preproc_imp.fit_transform(Xi_train)
Xi_test_prep  = preproc_imp.transform(Xi_test)

### 🎯 O Desafio
1. Treinem um `RandomForestClassifier` (ou `GradientBoostingClassifier`) no `Pipeline`.
2. Extraiam as **importâncias** (`.feature_importances_`) e liguem aos nomes das colunas
   geradas pelo one-hot (`get_feature_names_out`).
3. Plotem o **Top 15** e respondam: **é a região? o departamento? o modal?**

In [ ]:
# TODO — Pipeline com RandomForestClassifier(n_estimators=200, random_state=42) e treino
# TODO — recuperar nomes das features: modelo['prep'].get_feature_names_out()
# TODO — montar uma Series importancia x nome, ordenar e plotar o Top 15


---
## 🗺️ Parte 4 — Clustering Geográfico (K-Means)

Nesta etapa, vamos usar as coordenadas geográficas (Latitude e Longitude) para entender a distribuição espacial dos nossos pedidos e, mais importante, identificar onde estão os nossos maiores gargalos.

In [ ]:
# 1. Isolando as variáveis de interesse e removendo nulos
geo = df[['Latitude', 'Longitude', 'Delivery Status', 'Order Status']].dropna(
    subset=['Latitude', 'Longitude']).copy()

# 2. Criando as flags de risco (1 para sim, 0 para não)
geo['atraso'] = (geo['Delivery Status'] == 'Late delivery').astype(int)
geo['fraude'] = (geo['Order Status'] == 'SUSPECTED_FRAUD').astype(int)

# 3. Padronizando as coordenadas geográficas
coords = StandardScaler().fit_transform(geo[['Latitude', 'Longitude']])

print('Preparação concluída. Pontos geográficos:', geo.shape[0])

Chegou a sua vez de construir o mapa de risco! Siga os passos abaixo:

1. Rodem o **K-Means** (sugestão: `n_clusters=6`) sobre a matriz de coordenadas padronizadas (`coords`).
2. Adicionem os resultados no DataFrame `geo` e, para cada cluster, calculem a **taxa de atraso** e a **taxa de fraude**.
3. Plotem o mapa (um *scatter plot* de Longitude × Latitude colorido pelo cluster) e **identifiquem visualmente os clusters de maior risco**.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# TODO 1 — Instanciar e treinar o KMeans(n_clusters=6, random_state=42, n_init=10) sobre 'coords'
# km = ...
# geo['cluster'] = ...

# TODO 2 — Agrupar o dataframe 'geo' por cluster para ver: tamanho do grupo, média de 'atraso' e média de 'fraude'
# resumo = ...
# print(resumo)

# TODO 3 — Criar um scatter plot de Longitude (eixo X) x Latitude (eixo Y) colorido pela coluna 'cluster'
# plt.figure(figsize=(10, 6))
# ...
# plt.title('Mapa de Clusters Geográficos')
# plt.show()


---
## 🏁 Entregáveis

1. Classificador de atraso + matriz de confusão e a discussão FP × FN (Parte 1).
2. Regressão de `Sales` com R²/MAE/RMSE e o gráfico real×previsto (Parte 2).
3. Top variáveis do Random Forest e a interpretação de negócio (Parte 3).
4. Clusters geográficos + mapa de risco logístico (Parte 4).

Bom trabalho! 🚀